In [ ]:
##Advanced QC-Q Index

In [ ]:
#1.Qgaps-Temporal Consistency

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ==========================================
# CONFIGURATION
# ==========================================
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Directory containing daily files (e.g., from Basic QC)
DATA_DIR = BASE_DIR / "data" / "basic_qc_passed"

# Output: File Path
OUTPUT_DIR = BASE_DIR / "output" / "advanced_qc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUTPUT_DIR / "q_gaps_index_report.csv"

ID_COL   = "station_id"
RAIN_COL = "rain(mm)"

# ==========================================
# Utils
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Supports: 2022-02-03.csv, 20220203.csv, 2022_02_03.csv"""
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

def longest_consecutive_true(mask: pd.Series) -> int:
    """Returns the longest streak of True values (consecutive gaps)"""
    if mask.sum() == 0:
        return 0
    grp = (mask != mask.shift()).cumsum()
    return int(mask.groupby(grp).sum().max())

# ==========================================
# Load all daily files
# ==========================================
print("[+] Loading daily CSV files...")
rows = []
file_dates = []

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    print(f"[-] No CSV files found in: {DATA_DIR}")
    exit()

for fp in csv_files:
    date = parse_date_from_filename(fp)
    file_dates.append(date)
    df = pd.read_csv(fp)

    # Standardize columns
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain_mm"})
    
    # Convert to numeric; non-numeric values become NaN
    df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")
    
    # Negative values (impossible) become NaN
    df.loc[df["rain_mm"] < 0, "rain_mm"] = np.nan

    df["date"] = date
    rows.append(df[["station_id", "rain_mm", "date"]])

df_all = pd.concat(rows, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["date"])
df_all["year"] = df_all["date"].dt.year

# All dates where files exist
file_dates = pd.Index(sorted(set(file_dates)))
years_in_files = sorted(pd.unique(file_dates.year))

# Find first/last valid record (>=0) for each station
bounds = (
    df_all.loc[df_all["rain_mm"].notna()]
         .groupby("station_id")["date"]
         .agg(overall_start="min", overall_end="max")
         .reset_index()
)

all_stations = sorted(df_all["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# Prepare file date index by year
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# ==========================================
# Compute Qgaps Index
# ==========================================
print("[+] Computing Qgaps index...")
results = []

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]   # NaT if station has no valid values
    overall_end   = b["overall_end"]
    had_any_valid_ever = pd.notna(overall_start) and pd.notna(overall_end)

    g_sta = df_all.loc[df_all["station_id"] == sid].copy()

    for yr in years_in_files:
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        if len(fdy) == 0:
            continue  # No files for this year

        if had_any_valid_ever:
            # Operational window for current year: [max(start, Jan1), min(end, Dec31)]
            y_start = pd.Timestamp(year=yr, month=1, day=1)
            y_end   = pd.Timestamp(year=yr, month=12, day=31)
            w_start = max(overall_start, y_start)
            w_end   = min(overall_end, y_end)
            
            if w_start > w_end:
                continue  # Out of station's operational period

            # Use dates that match existing files within this window
            use_dates = fdy[(fdy >= w_start) & (fdy <= w_end)]
            n = int(len(use_dates))
            
            if n == 0:
                results.append({
                    "station_id": sid, "year": yr,
                    "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                    "expected_days": 0, "n_gap": np.nan, "Lmax_gap": np.nan,
                    "Qgaps": np.nan, "all_year_missing": np.nan,
                    "had_any_valid_ever": True
                })
                continue

            # Create daily series based on available files and map rainfall
            s = (g_sta.set_index("date")["rain_mm"]
                      .reindex(use_dates)
                      .astype(float))

            # Missing (NaN) is treated as a gap
            gap_mask = s.isna()
            n_gap = int(gap_mask.sum())
            Lmax_gap = longest_consecutive_true(gap_mask)

            # Qgaps Formula: 100 - 100 * (2*n_gap + Lmax_gap) / n
            Qgaps = 100.0 - 100.0 * (2 * n_gap + Lmax_gap) / n
            Qgaps = max(min(Qgaps, 100.0), 0.0)  # Clamp between 0 and 100

            all_year_missing = (n > 0 and n_gap == n)

            results.append({
                "station_id": sid, "year": yr,
                "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                "expected_days": n, "n_gap": n_gap, "Lmax_gap": Lmax_gap,
                "Qgaps": Qgaps, "all_year_missing": all_year_missing,
                "had_any_valid_ever": True
            })

        else:
            # Station never had any valid data: entire set of files is a gap
            n = int(len(fdy))
            n_gap = n
            Lmax_gap = n
            Qgaps = 100.0 - 100.0 * (2 * n_gap + Lmax_gap) / n
            Qgaps = max(min(Qgaps, 100.0), 0.0)

            results.append({
                "station_id": sid, "year": yr,
                "start_date_used": "", "end_date_used": "",
                "expected_days": n, "n_gap": n_gap, "Lmax_gap": Lmax_gap,
                "Qgaps": Qgaps, "all_year_missing": True,
                "had_any_valid_ever": False
            })

# ==========================================
# Export Results
# ==========================================
out = pd.DataFrame(results).sort_values(["year", "station_id"])
out.to_csv(OUT_PATH, index=False)

print(f"\n[+] QC Process Completed Successfully.")
print(f"    - Report Saved: {OUT_PATH}")
print(f"\nSample Data (Top 12):")
print(out.head(12))

In [ ]:
#2.P-Data Completeness

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# -----------------------------
# CONFIGURATION
# -----------------------------
# Using relative paths for GitHub portability
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Directory containing filtered daily files
DATA_DIR = BASE_DIR / "data" / "basic_qc_passed"

# Output: File Path
OUTPUT_DIR = BASE_DIR / "output" / "advanced_qc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUTPUT_DIR / "p_index_report.csv"

ID_COL   = "station_id"
RAIN_COL = "rain(mm)"

# -----------------------------
# Utilities
# -----------------------------
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Supports: 2022-02-03.csv, 20220203.csv, 2022_02_03.csv"""
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

# -----------------------------
# Load all daily files
# -----------------------------
print("[+] Loading daily CSV files...")
rows = []
file_dates = []

csv_files = sorted(DATA_DIR.glob("*.csv"))
if not csv_files:
    print(f"[-] No CSV files found in: {DATA_DIR}")
    exit()

for fp in csv_files:
    date = parse_date_from_filename(fp)
    file_dates.append(date)
    df = pd.read_csv(fp)

    # Standardize column names
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain_mm"})
    
    # Convert to numeric; non-numeric values become NaN
    df["rain_mm"] = pd.to_numeric(df["rain_mm"], errors="coerce")
    
    # Remove impossible negative values
    df.loc[df["rain_mm"] < 0, "rain_mm"] = np.nan

    df["date"] = date
    rows.append(df[["station_id", "rain_mm", "date"]])

# Combine all data into a single dataframe
all_data = pd.concat(rows, ignore_index=True)
all_data["date"] = pd.to_datetime(all_data["date"])
all_data["year"] = all_data["date"].dt.year

# All dates where actual files exist (e.g., 364 days)
file_dates = pd.Index(sorted(set(file_dates)))
years_in_files = sorted(pd.unique(file_dates.year))

# Determine operational start/end based on first/last valid non-NaN record per station
bounds = (
    all_data.loc[all_data["rain_mm"].notna()]
        .groupby("station_id")["date"]
        .agg(overall_start="min", overall_end="max")
        .reset_index()
)

# Ensure all stations present in the dataset are included
all_stations = sorted(all_data["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# Initialize result storage
results = []

# Prepare mapping of available file dates per year for expected_days count
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# -----------------------------
# Compute P Index (Data Availability)
# -----------------------------
print("[+] Computing P index (Data Availability)...")
for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]  # May be NaT if no valid data ever existed
    overall_end   = b["overall_end"]

    # Check if station ever had at least one valid record in the entire dataset
    had_any_valid_ever = pd.notna(overall_start) and pd.notna(overall_end)

    # Subset for current station
    g_sta = all_data.loc[all_data["station_id"] == sid].copy()

    for yr in years_in_files:
        # Get count of actual files for this specific year
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        expected_days = int(len(fdy))

        if expected_days == 0:
            continue # Skip if no files exist for this year

        if had_any_valid_ever:
            # Operational window for current year: [max(start, Jan1), min(end, Dec31)]
            y_start = pd.Timestamp(year=yr, month=1, day=1)
            y_end   = pd.Timestamp(year=yr, month=12, day=31)
            w_start = max(overall_start, y_start)
            w_end   = min(overall_end, y_end)
            
            if w_start > w_end:
                continue # Out of station's operational window for this year

            # Count expected files only within the operational window
            mask_fd = (fdy >= w_start) & (fdy <= w_end)
            exp_days_use = int(mask_fd.sum())

            # Count observations (n_obs) within the operational window
            mask_rows = (g_sta["date"] >= w_start) & (g_sta["date"] <= w_end)
            n_obs = int(g_sta.loc[mask_rows, "rain_mm"].notna().sum())

            # Handle case where window overlaps but no files are available in that range
            if exp_days_use == 0:
                p_index = np.nan
            else:
                # Calculate P Index (Percentage of availability)
                p_index = 100.0 * n_obs / exp_days_use

            all_year_missing = (exp_days_use > 0 and n_obs == 0)

            results.append({
                "station_id": sid,
                "year": yr,
                "start_date_used": w_start.date() if pd.notna(w_start) else "",
                "end_date_used": w_end.date() if pd.notna(w_end) else "",
                "expected_days": exp_days_use,
                "n_obs": n_obs,
                "p_index": p_index,
                "all_year_missing": all_year_missing,
                "had_any_valid_ever": True
            })

        else:
            # Station never had any valid records: P = 0.0 for every year with files
            n_obs = 0
            p_index = 0.0
            results.append({
                "station_id": sid,
                "year": yr,
                "start_date_used": "",
                "end_date_used": "",
                "expected_days": expected_days,
                "n_obs": n_obs,
                "p_index": p_index,
                "all_year_missing": True,
                "had_any_valid_ever": False
            })

# -----------------------------
# Export Results
# -----------------------------
out = pd.DataFrame(results).sort_values(["year", "station_id"])
out.to_csv(OUT_PATH, index=False)

print(f"\n[+] QC Process Completed Successfully.")
print(f"    - Report Saved: {OUT_PATH}")
print(f"\nSample Results (Top 12):")
print(out.head(12))

In [ ]:
#3.Qmzero-Descriptive Statistics Review

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ==========================================
# CONFIGURATION
# ==========================================
# Using relative paths for GitHub portability
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Directory containing filtered daily files
DATA_DIR = BASE_DIR / "data" / "basic_qc_passed"

# Output: Reporting Paths
OUTPUT_DIR = BASE_DIR / "output" / "advanced_qc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_Q_REPORT   = OUTPUT_DIR / "q_mzero_index_summary.csv"
SAVE_MON_DETAIL = OUTPUT_DIR / "q_mzero_monthly_details.csv"

# Global Column Mapping
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# Qmzero Policy
COV_THR = 0.90            # Monthly data coverage threshold (>= 90%)
EXCLUDE_MONTH_END = True  # Exclude the last day of each month from calculation

# ==========================================
# Utilities
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Supports: 2022-02-03.csv, 20220203.csv, 2022_02_03.csv"""
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

# ==========================================
# Load all daily files
# ==========================================
print("[+] Loading daily CSV files...")
rows = []
csv_files = sorted(DATA_DIR.glob("*.csv"))

if not csv_files:
    print(f"[-] No CSV files found in: {DATA_DIR}")
    exit()

for fp in csv_files:
    df = pd.read_csv(fp)

    # Standardize column names
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        print(f"[!] Skipping {fp.name}: Missing required columns.")
        continue
        
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})

    # Date handling: Use date column if exists, otherwise parse from filename
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)

    # Data Cleaning: Convert to numeric, negative values become NaN
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan

    rows.append(df[["station_id", "rain", "date"]])

df_all = pd.concat(rows, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["date"])

# --- EXCLUDE LAST DAY OF MONTH (Policy Implementation) ---
if EXCLUDE_MONTH_END:
    print("[i] Excluding month-end records from analysis.")
    df_all = df_all[~df_all["date"].dt.is_month_end].copy()

df_all["year"] = df_all["date"].dt.year

# Record actual file dates after filter
file_dates = pd.Index(sorted(df_all["date"].unique()))
years_in_files = sorted(df_all["year"].unique())

# Find operational bounds per station (after month-end filter)
bounds = (
    df_all.loc[df_all["rain"].notna()]
      .groupby("station_id")["date"]
      .agg(overall_start="min", overall_end="max")
      .reset_index()
)
all_stations = sorted(df_all["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# Map file dates per year for indexing
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# ==========================================
# Compute Qmzero & Monthly Details
# ==========================================
print("[+] Computing Qmzero index and monthly statistics...")
res_rows = []
mon_rows = []

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]
    overall_end   = b["overall_end"]
    had_any_valid = pd.notna(overall_start) and pd.notna(overall_end)

    g_sta = (
        df_all.loc[df_all["station_id"] == sid, ["date", "rain"]]
          .copy()
          .set_index("date")
          .sort_index()
    )

    for yr in years_in_files:
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        if len(fdy) == 0:
            continue

        if not had_any_valid:
            # Station never had valid data: m=0, m0=0, Qmzero=NaN
            res_rows.append({"station_id": sid, "year": yr, "m": 0, "m0": 0, "qm_zero": np.nan})
            continue

        # Year operational window
        y_start = pd.Timestamp(year=yr, month=1, day=1)
        y_end   = pd.Timestamp(year=yr, month=12, day=31)
        w_start = max(overall_start, y_start) if pd.notna(overall_start) else y_start
        w_end   = min(overall_end, y_end)     if pd.notna(overall_end)   else y_end
        
        if w_start > w_end:
            continue

        # Actual available file dates within year window
        use_dates = fdy[(fdy >= w_start) & (fdy <= w_end)]
        if len(use_dates) == 0:
            res_rows.append({"station_id": sid, "year": yr, "m": 0, "m0": 0, "qm_zero": np.nan})
            continue

        # Station rain series based on valid use_dates
        s = g_sta.reindex(use_dates)["rain"].astype(float)

        # --- Monthly Analysis (Excluding month-end as per filter) ---
        dfm = pd.DataFrame({"rain": s})
        dfm["month"] = dfm.index.to_period("M")

        # file_days_in_month = count of available days (after exclusion)
        mon_cnt = dfm.groupby("month").size()

        # days_with_value = count of non-missing (including 0.0)
        mon_nonnull = dfm.groupby("month")["rain"].apply(lambda x: x.notna().sum())

        # Coverage calculation
        mon_cov = mon_nonnull / mon_cnt

        # Eligible month criteria (>= 90% coverage)
        eligible = mon_cov >= COV_THR

        # Monthly cumulative rain (must have at least one valid day)
        mon_sum = dfm.groupby("month")["rain"].sum(min_count=1)

        # Is month entirely zero? (Has data AND sum is exactly 0.0)
        is_all_zero = (mon_nonnull > 0) & (mon_sum == 0.0)

        # Store monthly metadata
        mon_detail = pd.DataFrame({
            "station_id": sid,
            "year": yr,
            "month": mon_cnt.index.astype(str),
            "file_days_in_month": mon_cnt.values,
            "days_with_value": mon_nonnull.values,
            "coverage_ratio": mon_cov.values,
            "is_eligible": eligible.values,
            "month_sum_mm": mon_sum.values,
            "is_all_zero": is_all_zero.values
        })
        mon_rows.append(mon_detail)

        # Count m (eligible months) and m0 (eligible months with 0.0 rain)
        m  = int(eligible.sum())
        m0 = int((eligible & is_all_zero).sum())

        # Formula: Qmzero = 100 - 100 * (m0 / m)
        qm_zero = (100.0 - 100.0 * m0 / m) if m > 0 else np.nan
        res_rows.append({"station_id": sid, "year": yr, "m_eligible": m, "m0_all_zero": m0, "qm_zero": qm_zero})

# ==========================================
# Export Results
# ==========================================
res_df = pd.DataFrame(res_rows).sort_values(["year", "station_id"])
res_df.to_csv(SAVE_Q_REPORT, index=False)

if mon_rows:
    mon_df = pd.concat(mon_rows, ignore_index=True)
    mon_df = mon_df.sort_values(["station_id", "year", "month"])
    mon_df.to_csv(SAVE_MON_DETAIL, index=False)

print(f"\n[+] QC Process Completed Successfully.")
print(f"    - Index Summary Saved: {SAVE_Q_REPORT}")
print(f"    - Monthly Details Saved: {SAVE_MON_DETAIL}")
print(f"\nSample Index Results:")
print(res_df.head(10))

In [ ]:
#4.Qwzero-Descriptive Statistics Review

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ==========================================
# CONFIGURATION
# ==========================================
# Using relative paths for GitHub portability
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Directory containing filtered daily files
DATA_DIR = BASE_DIR / "data" / "basic_qc_passed"

# Output: Reporting Path
OUTPUT_DIR = BASE_DIR / "output" / "advanced_qc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PATH = OUTPUT_DIR / "q_wzero_index_report.csv"

# Column Mapping
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# Qwzero Thresholds
RAINY_DAY_THRESHOLD = 1.0     # Threshold to define a "Rainy Day" (mm)
MIN_RAINY_DAYS      = 20      # Min rainy days required per year to compute Qwzero

# ==========================================
# Utilities
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Supports: 2022-02-03.csv, 20220203.csv, 2022_02_03.csv"""
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

# ==========================================
# Load all daily files
# ==========================================
print("[+] Loading daily CSV files...")
rows = []
csv_files = sorted(DATA_DIR.glob("*.csv"))

if not csv_files:
    print(f"[-] No CSV files found in: {DATA_DIR}")
    exit()

for fp in csv_files:
    df = pd.read_csv(fp)

    # Basic column check
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        print(f"[!] Skipping {fp.name}: Missing required columns.")
        continue

    # Standardize column names
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})

    # Date handling
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)

    # Data Cleaning: Numeric conversion and negative value removal
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan

    rows.append(df[["station_id", "rain", "date"]])

df_all = pd.concat(rows, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["date"])
df_all["year"] = df_all["date"].dt.year

# Record actual file dates available
file_dates = pd.Index(sorted(df_all["date"].unique()))
years_in_files = sorted(df_all["year"].unique())
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# Find operational bounds per station (min/max date with valid rain)
bounds = (
    df_all.loc[df_all["rain"].notna()]
      .groupby("station_id")["date"]
      .agg(overall_start="min", overall_end="max")
      .reset_index()
)
all_stations = sorted(df_all["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# ==========================================
# Compute Qwzero Index
# ==========================================
print("[+] Computing Qwzero index based on Day of Week (DOW)...")
results = []
dow_names = {0: "Mon", 1: "Tue", 2: "Wed", 3: "Thu", 4: "Fri", 5: "Sat", 6: "Sun"}

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]
    overall_end   = b["overall_end"]
    had_any_valid = pd.notna(overall_start) and pd.notna(overall_end)

    g_sta = df_all.loc[df_all["station_id"] == sid, ["date", "rain"]].copy().set_index("date").sort_index()

    for yr in years_in_files:
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        if len(fdy) == 0:
            continue

        if not had_any_valid:
            # Station never had valid data: Index cannot be computed
            out = {
                "station_id": sid, "year": yr,
                "start_date_used": "", "end_date_used": "",
                "rainy_days_total": 0, "mean_by_dow": np.nan, "std_by_dow": np.nan,
                "cv_by_dow": np.nan, "qw_zero": np.nan
            }
            for i in range(7): out[f"rainy_{dow_names[i]}"] = 0
            results.append(out)
            continue

        # operational year window
        y_start = pd.Timestamp(year=yr, month=1, day=1)
        y_end   = pd.Timestamp(year=yr, month=12, day=31)
        w_start = max(overall_start, y_start)
        w_end   = min(overall_end, y_end)
        
        if w_start > w_end:
            continue

        # Available file dates within operational window
        use_dates = fdy[(fdy >= w_start) & (fdy <= w_end)]
        if len(use_dates) == 0:
            out = {
                "station_id": sid, "year": yr,
                "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                "rainy_days_total": 0, "mean_by_dow": np.nan, "std_by_dow": np.nan,
                "cv_by_dow": np.nan, "qw_zero": np.nan
            }
            for i in range(7): out[f"rainy_{dow_names[i]}"] = 0
            results.append(out)
            continue

        # Station rain series for current period
        s = g_sta.reindex(use_dates)["rain"].astype(float)

        # Define "Rainy Day" mask
        rainy_mask = (s >= RAINY_DAY_THRESHOLD)
        rainy_days_total = int(rainy_mask.sum())

        # Check for minimum rainy days criteria
        if rainy_days_total < MIN_RAINY_DAYS:
            out = {
                "station_id": sid, "year": yr,
                "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                "rainy_days_total": rainy_days_total,
                "mean_by_dow": np.nan, "std_by_dow": np.nan,
                "cv_by_dow": np.nan, "qw_zero": np.nan
            }
            # Fill daily counts even if criteria isn't met
            counts = pd.Series(0, index=range(7))
            if rainy_days_total > 0:
                dows = pd.Index(use_dates).weekday
                counts = pd.Series(rainy_mask.astype(int).values, index=dows).groupby(level=0).sum().reindex(range(7), fill_value=0)
            for i in range(7): out[f"rainy_{dow_names[i]}"] = int(counts[i])
            results.append(out)
            continue

        # Group rainy days by Day of Week (0=Mon...6=Sun)
        dows = pd.Index(use_dates).weekday
        counts = pd.Series(rainy_mask.astype(int).values, index=dows).groupby(level=0).sum().reindex(range(7), fill_value=0)

        mean_val = float(counts.mean())
        std_val  = float(counts.std(ddof=0))
        cv_val   = (std_val / mean_val) if mean_val > 0 else np.nan
        
        # Formula: Qwzero = 100 - 100 * CV
        qw_zero = 100.0 - 100.0 * cv_val if pd.notna(cv_val) else np.nan
        if pd.notna(qw_zero):
            qw_zero = max(min(qw_zero, 100.0), 0.0) # Clamp between 0 and 100

        out = {
            "station_id": sid, "year": yr,
            "start_date_used": w_start.date(), "end_date_used": w_end.date(),
            "rainy_days_total": rainy_days_total,
            "mean_by_dow": mean_val, "std_by_dow": std_val, "cv_by_dow": cv_val, "qw_zero": qw_zero
        }
        for i in range(7): out[f"rainy_{dow_names[i]}"] = int(counts[i])
        results.append(out)

# ==========================================
# Export Results
# ==========================================
res_df = pd.DataFrame(results).sort_values(["year", "station_id"])
res_df.to_csv(SAVE_PATH, index=False)

print(f"\n[+] QC Process Completed Successfully.")
print(f"    - Index Report Saved: {SAVE_PATH}")
print(f"\nSample Results (Top 12):")
print(res_df.head(12))

In [ ]:
#5.Qoutliers-Outlier Detection

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re

# ==========================================
# CONFIGURATION
# ==========================================
# Using relative paths for GitHub portability
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: Directory containing daily files (after Basic QC)
DATA_DIR = BASE_DIR / "data" / "basic_qc_passed"

# Output: Reporting Paths
OUTPUT_DIR = BASE_DIR / "output" / "advanced_qc"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_YR_REPORT  = OUTPUT_DIR / "q_outliers_index_summary.csv"
SAVE_MON_DETAIL = OUTPUT_DIR / "q_outliers_monthly_details.csv"

# Column Mapping
ID_COL   = "station_id"
DATE_COL = "date"
RAIN_COL = "rain(mm)"

# Thresholds
RAINY_DAY_THRESHOLD = 1.0  # Rain >= 1 mm defined as a "Rainy Day"

# ==========================================
# Utilities
# ==========================================
def parse_date_from_filename(path: Path) -> pd.Timestamp:
    """Supports: 2022-02-03.csv, 20220203.csv, 2022_02_03.csv"""
    s = path.stem
    m = re.search(r'(\d{4})[-_]?(\d{2})[-_]?(\d{2})', s)
    if not m:
        raise ValueError(f"Cannot parse date from filename: {path.name}")
    return pd.Timestamp(int(m.group(1)), int(m.group(2)), int(m.group(3)))

# ==========================================
# Load all daily files
# ==========================================
print("[+] Loading daily CSV files...")
rows = []
csv_files = sorted(DATA_DIR.glob("*.csv"))

if not csv_files:
    print(f"[-] No CSV files found in: {DATA_DIR}")
    exit()

for fp in csv_files:
    df = pd.read_csv(fp)

    # Basic column validation
    if ID_COL not in df.columns or RAIN_COL not in df.columns:
        print(f"[!] Skipping {fp.name}: Missing required columns.")
        continue

    # Standardize names
    df = df.rename(columns={ID_COL: "station_id", RAIN_COL: "rain"})

    # Date handling
    if DATE_COL in df.columns:
        df["date"] = pd.to_datetime(df[DATE_COL], errors="coerce")
    else:
        df["date"] = parse_date_from_filename(fp)

    # Data Cleaning
    df["rain"] = pd.to_numeric(df["rain"], errors="coerce")
    df.loc[df["rain"] < 0, "rain"] = np.nan

    rows.append(df[["station_id", "rain", "date"]])

df_all = pd.concat(rows, ignore_index=True)
df_all["date"] = pd.to_datetime(df_all["date"])
df_all["year"] = df_all["date"].dt.year

# Mapping available file dates
file_dates = pd.Index(sorted(df_all["date"].unique()))
years_in_files = sorted(df_all["year"].unique())
file_dates_by_year = {yr: file_dates[file_dates.year == yr] for yr in years_in_files}

# Find operational bounds per station
bounds = (
    df_all.loc[df_all["rain"].notna()]
      .groupby("station_id")["date"]
      .agg(overall_start="min", overall_end="max")
      .reset_index()
)
all_stations = sorted(df_all["station_id"].unique())
bounds = pd.merge(pd.DataFrame({"station_id": all_stations}),
                  bounds, on="station_id", how="left")

# ==========================================
# Qoutliers Calculation
# ==========================================
print("[+] Detecting outliers and computing Qoutliers index...")
yr_rows = []
mon_rows = []

for sid in all_stations:
    b = bounds.loc[bounds["station_id"] == sid].iloc[0]
    overall_start = b["overall_start"]
    overall_end   = b["overall_end"]
    had_any_valid = pd.notna(overall_start) and pd.notna(overall_end)

    g_sta = df_all.loc[df_all["station_id"] == sid, ["date", "rain"]].copy().set_index("date").sort_index()

    for yr in years_in_files:
        fdy = file_dates_by_year.get(yr, pd.Index([]))
        if len(fdy) == 0:
            continue

        if not had_any_valid:
            yr_rows.append({
                "station_id": sid, "year": yr,
                "start_date_used": "", "end_date_used": "",
                "n_obs": 0, "n_outliers": 0, "q_outliers": np.nan
            })
            continue

        # operational year window
        y_start = pd.Timestamp(year=yr, month=1, day=1)
        y_end   = pd.Timestamp(year=yr, month=12, day=31)
        w_start = max(overall_start, y_start)
        w_end   = min(overall_end, y_end)
        
        if w_start > w_end:
            continue

        use_dates = fdy[(fdy >= w_start) & (fdy <= w_end)]
        if len(use_dates) == 0:
            yr_rows.append({
                "station_id": sid, "year": yr,
                "start_date_used": w_start.date(), "end_date_used": w_end.date(),
                "n_obs": 0, "n_outliers": 0, "q_outliers": np.nan
            })
            continue

        # Extract daily series for station
        s = g_sta.reindex(use_dates)["rain"].astype(float)
        n_obs = int(s.notna().sum())

        # Monthly processing for outlier detection (IQR method)
        monthly_data = s.to_frame("rain")
        monthly_data["month"] = monthly_data.index.to_period("M")

        total_n_out = 0
        monthly_metadata = []

        for mth, sub in monthly_data.groupby("month"):
            vals = sub["rain"].dropna()
            obs_days = int(vals.size)
            rainy_vals = vals[vals >= RAINY_DAY_THRESHOLD] # Filter for rainy days only

            if rainy_vals.size == 0:
                # Cannot determine distribution (T) without rainy days
                q1 = q3 = iqr = thr = np.nan
                out_days = 0
            else:
                q1 = float(rainy_vals.quantile(0.25))
                q3 = float(rainy_vals.quantile(0.75))
                iqr = q3 - q1
                thr = q3 + 3.0 * iqr # Using 3.0 for extreme outliers
                out_days = int((vals > thr).sum())

            total_n_out += out_days

            monthly_metadata.append({
                "station_id": sid,
                "year": yr,
                "month": str(mth),
                "obs_days_in_month": obs_days,
                "rainy_days_in_month": int(rainy_vals.size),
                "q1": q1, "q3": q3, "iqr": iqr, "threshold_t": thr,
                "outlier_days_in_month": out_days
            })

        # Calculate Qoutliers Index: 100 * (1 - N_out / N_obs)
        if n_obs > 0:
            q_outliers = 100.0 * (1.0 - total_n_out / n_obs)
            q_outliers = max(min(q_outliers, 100.0), 0.0) # Clamp 0-100
        else:
            q_outliers = np.nan

        yr_rows.append({
            "station_id": sid, "year": yr,
            "start_date_used": w_start.date(), "end_date_used": w_end.date(),
            "n_obs": n_obs, "n_outliers": total_n_out, "q_outliers": q_outliers
        })
        if monthly_metadata:
            mon_rows.extend(monthly_metadata)

# ==========================================
# Export Results
# ==========================================
yr_df = pd.DataFrame(yr_rows).sort_values(["year", "station_id"])
yr_df.to_csv(SAVE_YR_REPORT, index=False)

if mon_rows:
    mon_df = pd.DataFrame(mon_rows).sort_values(["station_id", "year", "month"])
    mon_df.to_csv(SAVE_MON_DETAIL, index=False)

print(f"\n[+] Outlier Analysis Completed.")
print(f"    - Summary Report: {SAVE_YR_REPORT}")
print(f"    - Monthly Detail: {SAVE_MON_DETAIL}")
print(f"\nSample Results (Top 10):")
print(yr_df.head(10))

In [ ]:
#6.Q-Index

In [ ]:
your-repo-name/
│
├── data/
│   └── basic_qc_passed/       <-- (Input) Daily CSV files
├── output/
│   └── advanced_qc/           <-- (Output) All QC results
│       ├── p_index_report.csv
│       ├── q_gaps_index_summary.csv
│       ├── q_mzero_index_summary.csv
│       ├── q_wzero_index_report.csv
│       ├── q_outliers_index_summary.csv
│       └── final_q_index_aggregation.xlsx
└── q_aggregation_master.py    <-- Script ตัวนี้

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================
# Using relative paths for GitHub portability
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Target Directory (Where the 5 index CSVs are stored)
QC_OUTPUT_DIR = BASE_DIR / "output" / "advanced_qc"

# Input File (Assuming you have a combined Excel or CSV of all sub-indices)
# If you have separate files, you would typically merge them first.
# Here we assume the master file for aggregation is located in QC_OUTPUT_DIR.
INPUT_FILE = QC_OUTPUT_DIR / "combined_indices_template.xlsx" 

# Output File Path
OUTPUT_FILE = QC_OUTPUT_DIR / "final_q_index_aggregation.xlsx"

# Key columns and Index Terms
ID_COLS = ["station_id", "year"]
TERMS = ["p_index", "q_gaps", "qm_zero", "qw_zero", "q_outliers"]

# ==========================================
# Load and Process Data
# ==========================================
if not INPUT_FILE.exists():
    print(f"[-] Input file not found: {INPUT_FILE}")
    print("[i] Note: Ensure you have merged P, Qgaps, Qmzero, Qwzero, and Qoutliers into one file first.")
    exit()

print(f"[+] Loading indices for aggregation: {INPUT_FILE.name}")
df = pd.read_excel(INPUT_FILE)

# Ensure all required columns exist
missing_cols = [c for c in TERMS if c not in df.columns]
if missing_cols:
    # If using CSV names from previous scripts, update mapping here
    print(f"[!] Warning: Missing columns {missing_cols}. Attempting to match case-insensitive...")
    # Basic mapping for compatibility with previous scripts' outputs
    df.columns = [c.lower() for c in df.columns]

# Convert terms to numeric; invalid entries become NaN
df[TERMS] = df[TERMS].apply(pd.to_numeric, errors="coerce")

# Clip values to 0-100 range to handle potential data entry errors
df[TERMS] = df[TERMS].clip(lower=0, upper=100)

# ==========================================
# Compute Aggregate Q-Index
# ==========================================
print("[+] Calculating aggregated Q-Index...")

# 1. Count available indices per row (ignores NaN)
df["terms_used"] = df[TERMS].notna().sum(axis=1)

# 2. Compute Q: Dynamic average based on available terms
# (Calculates sum of available terms / number of available terms)
sum_terms = df[TERMS].sum(axis=1, skipna=True)
df["Q_score"] = sum_terms / df["terms_used"].replace({0: np.nan})

# 3. Optional: Q_min3 (Only valid if at least 3 indices are present)
df["Q_min3"] = np.where(df["terms_used"] >= 3, df["Q_score"], np.nan)

# 4. Optional: Q_strict5 (Only valid if all 5 indices are present)
# Using mean with skipna=False returns NaN if any term is missing
df["Q_strict5"] = df[TERMS].mean(axis=1, skipna=False)

# Rounding results for readability
result_cols = ["Q_score", "Q_min3", "Q_strict5"]
df[result_cols] = df[result_cols].round(3)

# ==========================================
# Export and Formatting
# ==========================================
# Reorder columns: IDs first, then sub-indices, then final Q results
existing_ids = [c for c in ID_COLS if c in df.columns]
other_cols = [c for c in df.columns if c not in existing_ids + TERMS + ["terms_used"] + result_cols]

final_column_order = existing_ids + TERMS + ["terms_used"] + result_cols + other_cols
df = df[final_column_order]

df.to_excel(OUTPUT_FILE, index=False)

print(f"\n[+] Aggregation Completed Successfully.")
print(f"    - Final File Saved: {OUTPUT_FILE}")
print(f"\nSummary of Aggregated Q (Top 10):")
print(df[existing_ids + result_cols].head(10))

In [ ]:
#7.Q-Index Cut-off

In [ ]:
your-repo-name/
│
├── data/
│   └── basic_qc_passed/       <-- (Input) Daily CSV files (364 files)
├── output/
│   └── advanced_qc/           
│       ├── final_q_index_aggregation.xlsx   <-- (Input) The Q-Index file
│       └── cleaned_daily_data/              <-- (Output) Cleaned CSVs will be here
└── clean_low_q_stations.py    <-- This script

In [ ]:
import pandas as pd
from pathlib import Path

# ==========================================
# CONFIGURATION
# ==========================================
# Using relative paths for GitHub portability
BASE_DIR = Path(__file__).parent if '__file__' in locals() else Path.cwd()

# Input: The Excel file containing aggregated Q results
EXCEL_PATH = BASE_DIR / "output" / "advanced_qc" / "final_q_index_aggregation.xlsx"

# Input: Directory containing daily CSV files to be cleaned
CSV_INPUT_DIR = BASE_DIR / "data" / "basic_qc_passed"

# Output: Directory where cleaned CSVs will be saved
OUT_DIR = BASE_DIR / "output" / "advanced_qc" / "cleaned_daily_data"

# Column Mapping
ID_COL = "station_id"
Q_COL  = "Q_score"  # Using the aggregate score column from the previous script

# Settings
DRY_RUN = False  # Set to True to simulate without saving files
Q_THRESHOLD = 50.0

# ==========================================
# Utilities
# ==========================================
def normalize_id_series(s: pd.Series) -> pd.Series:
    """Standardizes IDs: string format, trimmed, and removes trailing '.0'"""
    return (s.astype(str)
             .str.strip()
             .str.replace(r"\.0$", "", regex=True))

# ==========================================
# 1. Load Q-Index Data
# ==========================================
if not EXCEL_PATH.exists():
    raise FileNotFoundError(f"[-] Aggregated Q file not found: {EXCEL_PATH}")

df_q = pd.read_excel(EXCEL_PATH)

if ID_COL not in df_q.columns or Q_COL not in df_q.columns:
    raise ValueError(f"[-] Excel must contain columns '{ID_COL}' and '{Q_COL}'.")

# Identify station_ids where Q < threshold
# Numeric conversion ensures comparison works even with 'missing' strings
low_q_mask = pd.to_numeric(df_q[Q_COL], errors="coerce") < Q_THRESHOLD
low_q_ids = normalize_id_series(df_q.loc[low_q_mask, ID_COL]).unique()
low_q_ids = set([x for x in low_q_ids if x != "nan"])

print(f"[+] Found {len(low_q_ids)} stations with Q < {Q_THRESHOLD}")

# ==========================================
# 2. Process and Clean CSVs
# ==========================================
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(CSV_INPUT_DIR.glob("*.csv"))
if not csv_files:
    raise RuntimeError(f"[-] No CSV files found in: {CSV_INPUT_DIR}")

total_removed_rows = 0
total_initial_rows = 0

print(f"[+] Starting cleaning process...")

for fp in csv_files:
    df = pd.read_csv(fp)
    
    if ID_COL not in df.columns:
        print(f" [!] Skipping {fp.name}: Column '{ID_COL}' missing.")
        continue

    # Temporary normalization for comparison
    df["_id_norm"] = normalize_id_series(df[ID_COL])
    rows_before = len(df)

    # Filter out Low-Q stations
    mask_to_remove = df["_id_norm"].isin(low_q_ids)
    removed_count = int(mask_to_remove.sum())
    
    # Keep only records NOT in the low_q_ids list
    cleaned_df = df.loc[~mask_to_remove].drop(columns=["_id_norm"])

    total_removed_rows += removed_count
    total_initial_rows += rows_before

    if DRY_RUN:
        print(f" [DRY RUN] {fp.name}: Removed {removed_count} rows, Kept {len(cleaned_df)}")
    else:
        cleaned_df.to_csv(OUT_DIR / fp.name, index=False)
        print(f" [+] {fp.name}: Removed {removed_count} rows, Kept {len(cleaned_df)}")

# ==========================================
# Summary
# ==========================================
print("-" * 30)
print(f"Summary of Cleaning (Q < {Q_THRESHOLD}):")
print(f" - Files Processed: {len(csv_files)}")
print(f" - Total Rows Removed: {total_removed_rows}")
print(f" - Total Rows Remaining: {total_initial_rows - total_removed_rows}")
print(f" - Output Directory: {OUT_DIR}")
if DRY_RUN:
    print(" [!] This was a DRY RUN. No files were actually saved.")